# HyLeakAI — U-Net surrogate training (Colab)

Trains the U-Net-Small surrogate that maps geology and operating stage to
H2 saturation and reservoir pressure fields, reproducing Mao et al. (2025).

**Before running:** Runtime → Change runtime type → **T4 GPU**. The
hardware check below will refuse to continue on CPU rather than silently
start a run that would take days.

**Why Small and not Large.** The paper reports U-Net-Small *with cyclic and
distance channels* at 8.6% pressure test error — level with U-Net-Large's
8.61% at 124M parameters and 35 GB. Without those channels Small collapses to
32.7%. The extra inputs, not the parameter count, buy the accuracy. So this
runs a 7.7M-parameter model on a free T4 rather than a 124M-parameter one on
an A100, and gives up almost nothing.

**Checkpoints go to Google Drive every epoch.** Colab sessions disconnect;
resuming is expected, not exceptional.

In [ ]:
# 1. Hardware check — fail loudly rather than train on CPU by accident.
import torch, subprocess

assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU. "
    "Training on a Colab CPU would take days; use the --sim-limit CPU "
    "fallback locally instead."
)
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"GPU: {name}  ({vram:.1f} GiB)")
print(f"torch {torch.__version__}")

# U-Net-Small needs ~7.5 GB at batch 128; batch 32 fits comfortably in 15 GB.
BATCH_SIZE = 64 if vram > 20 else 32
print(f"Using batch size {BATCH_SIZE}")

In [ ]:
# 2. Mount Drive and locate the converted arrays.
#
# Upload these from the local machine after running src/data/lmdb_convert.py:
#     constants.npy   (1000, 2, 128, 128)      float32   ~131 MB
#     states.npy      (1000, 60, 3, 128, 128)  float16   ~5.9 GB
#     stats.json
#
# The raw 12.4 GB LMDB is NOT needed here. If Drive space is tight, upload a
# 2-channel states.npy (pressure + saturation only, ~3.9 GB) — the third
# channel is unused for training.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/hileak'
DATA_DIR = f'{DRIVE_DIR}/data'
CKPT_DIR = f'{DRIVE_DIR}/checkpoints'

import os, json
os.makedirs(CKPT_DIR, exist_ok=True)
for f in ('constants.npy', 'states.npy', 'stats.json'):
    p = f'{DATA_DIR}/{f}'
    assert os.path.exists(p), f'missing {p}'
    print(f'{f:16s} {os.path.getsize(p) / 2**20:9.1f} MiB')
print()
print(json.dumps(json.load(open(f'{DATA_DIR}/stats.json'))['stats'], indent=2)[:600])

In [ ]:
# 3. Get the code.
#    Either clone the repo, or upload the src/ tree to Drive and copy it in.
REPO_URL = ''  # e.g. 'https://github.com/<user>/hileak.git'

import shutil, sys, os
if REPO_URL:
    !git clone -q $REPO_URL /content/hileak
else:
    shutil.copytree(f'{DRIVE_DIR}/src', '/content/hileak/src', dirs_exist_ok=True)

os.chdir('/content/hileak')
sys.path.insert(0, '/content/hileak')

# Reading 5.9 GB off Drive is slow and the mount stalls under a DataLoader.
# Copy to local SSD first — this pays for itself within one epoch.
os.makedirs('/content/data', exist_ok=True)
for f in ('constants.npy', 'states.npy', 'stats.json'):
    if not os.path.exists(f'/content/data/{f}'):
        print(f'copying {f} to local disk...')
        shutil.copy(f'{DATA_DIR}/{f}', f'/content/data/{f}')
print('data staged locally')

In [ ]:
# 4. Architecture check against the paper's Table 1.
#    A mismatch here means any accuracy comparison to the paper is meaningless,
#    so this runs before a single training step.
from src.models.unet import assert_paper_parameter_counts
assert_paper_parameter_counts()

In [ ]:
# 5. Overfit test: 4 simulations, expect both losses -> ~0.
#    Catches channel-ordering, normalisation and architecture bugs in minutes,
#    before committing hours to the real run. Do not skip this.
!python -m src.train_unet --overfit 4 --epochs 40 \
    --data-dir /content/data --checkpoint-dir {CKPT_DIR} \
    --batch-size 16 --workers 2

In [ ]:
# 6. Full training. Resumable — rerun this cell verbatim after a disconnect
#    and it picks up from the last checkpoint.
import os
RESUME = f'{CKPT_DIR}/unet_small_last.pt'
resume_flag = f'--resume {RESUME}' if os.path.exists(RESUME) else ''
print('resuming' if resume_flag else 'starting fresh')

!python -m src.train_unet --size small --epochs 120 \
    --batch-size {BATCH_SIZE} --learning-rate 1e-4 --weight-decay 1e-5 \
    --lr-halve-every 50 --amp --workers 2 \
    --data-dir /content/data --checkpoint-dir {CKPT_DIR} {resume_flag}

In [ ]:
# 7. Held-out TEST error. Run once, at the end.
#    Asserts split disjointness before reporting anything.
!python -m src.train_unet --eval-only {CKPT_DIR}/unet_small_best.pt \
    --data-dir /content/data --batch-size {BATCH_SIZE} --workers 2

In [ ]:
# 8. Training curves, and the qualitative signatures from the paper.
#
#    Two signatures must appear or the model is wrong regardless of its
#    headline number (paper Figures 8 and 10):
#      - saturation error concentrates at the PLUME FRONT
#      - pressure error SPIKES at each injection -> withdrawal transition
import json
import matplotlib.pyplot as plt
import numpy as np
import torch

from src import config as C
from src.data.dataset import UHSDataset, Normalizer
from src.models.unet import build_unet, relative_l2

history = json.load(open(f'{CKPT_DIR}/unet_small_history.json'))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ep = [h['epoch'] for h in history]
ax[0].plot(ep, [h['train_pressure'] for h in history], label='train')
ax[0].plot(ep, [h['val_pressure'] for h in history], label='val')
ax[0].axhline(0.0861, ls='--', c='k', lw=1, label='paper U-Net-Large')
ax[0].set_title('pressure relative L2'); ax[0].set_xlabel('epoch'); ax[0].legend()
ax[1].plot(ep, [h['train_saturation'] for h in history], label='train')
ax[1].plot(ep, [h['val_saturation'] for h in history], label='val')
ax[1].axhline(0.0577, ls='--', c='k', lw=1, label='paper U-Net-Large')
ax[1].set_title('saturation relative L2'); ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# 9. Error versus timestep — the pressure spike test.
device = torch.device('cuda')
ckpt = torch.load(f'{CKPT_DIR}/unet_small_best.pt', map_location=device, weights_only=False)
meta = ckpt.get('meta', {})
normalizer = Normalizer.from_file('/content/data/stats.json')
test_ids = C.simulation_splits()['test'][:40]
ds = UHSDataset('test', data_dir='/content/data', normalizer=normalizer,
                use_cyclic=meta.get('use_cyclic', True),
                use_distance=meta.get('use_distance', True), sim_ids=test_ids)
model = build_unet(meta.get('size', 'small'), in_channels=ds.in_channels, out_channels=2).to(device)
model.load_state_dict(ckpt['model']); model.eval()

err_p = np.zeros(C.N_TIMESTEPS); err_s = np.zeros(C.N_TIMESTEPS)
with torch.no_grad():
    for t in range(1, C.N_TIMESTEPS + 1):
        x = torch.stack([ds.build_input_tensor(s, t) for s in test_ids]).to(device)
        y = torch.stack([torch.from_numpy(ds.build_target(s, t)) for s in test_ids]).to(device)
        p = model(x)
        err_p[t - 1] = relative_l2(p[:, 0], y[:, 0]).item()
        err_s[t - 1] = relative_l2(p[:, 1], y[:, 1]).item()

steps = np.arange(1, C.N_TIMESTEPS + 1)
transitions = [t for t in steps if C.cycle_index(t) == -1 and C.cycle_index(t - 1) == 1]
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(steps, err_p, label='pressure', marker='o', ms=3)
ax.plot(steps, err_s, label='saturation', marker='s', ms=3)
for i, t in enumerate(transitions):
    ax.axvline(t, c='r', ls=':', lw=1, label='injection -> withdrawal' if i == 0 else None)
ax.set_xlabel('timestep (2 months each)'); ax.set_ylabel('relative L2')
ax.set_title('Temporal coherence — expect pressure spikes at the red lines (paper Fig. 8)')
ax.legend(); plt.tight_layout(); plt.show()

at_trans = err_p[[t - 1 for t in transitions]].mean()
elsewhere = err_p[[t - 1 for t in steps if t not in transitions]].mean()
print(f'pressure error at transitions {at_trans:.4f} vs elsewhere {elsewhere:.4f} '
      f'({at_trans / elsewhere:.2f}x)')
print('PASS: reproduces the paper' if at_trans > elsewhere else
      'CHECK: the paper reports a clear spike here; its absence is suspicious')

In [ ]:
# 10. Field comparison — the plume-front error test (paper Fig. 10).
sim = test_ids[0]
show = [1, 3, 6, 12, 30, 60]
fig, axes = plt.subplots(4, len(show), figsize=(3 * len(show), 11))
with torch.no_grad():
    for j, t in enumerate(show):
        x = ds.build_input_tensor(sim, t)[None].to(device)
        y = ds.build_target(sim, t)
        p = model(x)[0].cpu().numpy()
        for row, (truth, pred, label) in enumerate([
            (y[1], p[1], 'saturation'), (y[0], p[0], 'pressure')]):
            axes[2 * row, j].imshow(truth, cmap='viridis')
            axes[2 * row, j].set_title(f'{label} truth  t={t}', fontsize=9)
            im = axes[2 * row + 1, j].imshow(truth - pred, cmap='RdBu_r')
            axes[2 * row + 1, j].set_title(f'{label} error', fontsize=9)
            plt.colorbar(im, ax=axes[2 * row + 1, j], fraction=0.046)
for a in axes.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.suptitle(f'Simulation {sim} — saturation error should ring the plume front')
plt.tight_layout(); plt.show()